# Build an Agent

[Tutorial](https://python.langchain.com/docs/tutorials/agents/#conclusion)

## Setup 

In [1]:
import getpass
import os

os.environ["TAVILY_API_KEY"] = getpass.getpass()

 ········


## Define the tools

We will use Tavily (a search engine) as a tool. 

We can use the dedicated langchain-tavily integration package to easily use Tavily search engine as tool with LangChain.    

In [2]:
from langchain_tavily import TavilySearch

search = TavilySearch(max_results=2)
search_results = search.invoke("What is the weather in SF")
print(search_results)
# If we want, we can create other tools.
# Once we have all the tools we want, we can put them in a list that we will reference later.
tools = [search]

{'query': 'What is the weather in SF', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in San Francisco', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1757852094, 'localtime': '2025-09-14 05:14'}, 'current': {'last_updated_epoch': 1757851200, 'last_updated': '2025-09-14 05:00', 'temp_c': 15.6, 'temp_f': 60.1, 'is_day': 0, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/night/116.png', 'code': 1003}, 'wind_mph': 4.9, 'wind_kph': 7.9, 'wind_degree': 244, 'wind_dir': 'WSW', 'pressure_mb': 1014.0, 'pressure_in': 29.93, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 90, 'cloud': 50, 'feelslike_c': 15.6, 'feelslike_f': 60.1, 'windchill_c': 14.2, 'windchill_f': 57.5, 'heatindex_c': 14.9, 'heatindex_f': 58.8, 'dewpoint_c': 14.3, 

## Using Language Models

In [3]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

from langchain.chat_models import init_chat_model

model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

Enter API key for Google Gemini:  ········


In [4]:
query = "Hi!"
response = model.invoke([{"role": "user", "content": query}])
response.text()

'Hi there! How can I help you today?'

We can now see what it is like to enable this model to do tool calling

In [5]:
model_with_tools = model.bind_tools(tools)

In [6]:
query = "Hi!"
response = model_with_tools.invoke([{"role": "user", "content": query}])

print(f"Message content: {response.text()}\n")
print(f"Tool calls: {response.tool_calls}")

Message content: Hello! How can I help you today?

Tool calls: []


In [7]:
query = "Search for the weather in SF"
response = model_with_tools.invoke([{"role": "user", "content": query}])

print(f"Message content: {response.text()}\n")
print(f"Tool calls: {response.tool_calls}")

Message content: 

Tool calls: [{'name': 'tavily_search', 'args': {'query': 'weather in SF'}, 'id': '9cddd250-31d9-489e-bc5d-b561c289fd16', 'type': 'tool_call'}]


Somehow I do not get the content of the message => to debug later

## Create the agent

There is a high and low level interface to create an agent. 

In [9]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(model, tools)

## Run the agent

Note that for now, these are all stateless queries.

In [10]:
input_message = {"role": "user", "content": "Hi!"}
response = agent_executor.invoke({"messages": [input_message]})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hi!
================================== Ai Message ==================================

Hello! How can I help you today?


In [12]:
input_message = {"role": "user", "content": "Search for the weather in SF"}
response = agent_executor.invoke({"messages": [input_message]})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Search for the weather in SF
================================== Ai Message ==================================
Tool Calls:
  tavily_search (acad09d0-e142-4769-91fe-006ef9892303)
 Call ID: acad09d0-e142-4769-91fe-006ef9892303
  Args:
    query: weather in SF
================================= Tool Message =================================
Name: tavily_search

{"query": "weather in SF", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "Weather in San Francisco", "url": "https://www.weatherapi.com/", "content": "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1757855359, 'localtime': '2025-09-14 06:09'}, 'current': {'last_updated_epoch': 1757854800, 'last_updated': '2025-09-14 06:00', 'temp_c': 16.1, 'temp_f': 61.0, 'is_day': 0, 'condition': {

## Streaming Messages

We've seen how the agent can be called with .invoke to get a final response. If the agent executes multiple steps, this may take a while. To show intermediate progress, we can stream back messages as they occur.

In [14]:
for step in agent_executor.stream({"messages": [input_message]}, stream_mode="values"):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Search for the weather in SF
================================== Ai Message ==================================
Tool Calls:
  tavily_search (881c083c-17ff-4c2e-880a-9aee8ff84b31)
 Call ID: 881c083c-17ff-4c2e-880a-9aee8ff84b31
  Args:
    query: weather in San Francisco
================================= Tool Message =================================
Name: tavily_search

{"query": "weather in San Francisco", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "Weather in San Francisco", "url": "https://www.weatherapi.com/", "content": "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1757855359, 'localtime': '2025-09-14 06:09'}, 'current': {'last_updated_epoch': 1757854800, 'last_updated': '2025-09-14 06:00', 'temp_c': 16.1, 'temp_f': 61.0, 'is_d

## Streaming tokens

In addition to streaming back messages, it is also useful to stream back tokens.

In [20]:

config = {"configurable": {"thread_id": "abc123"}}

for step, metadata in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="messages"
):
    if metadata["langgraph_node"] == "agent" and (text := step.text()):
        print(text, end="|\n")

The weather in San|
 Francisco on Sunday, September 14, 2025, will be partly cloudy|
 with a temperature of 61.0°F (16.1°C) at 6:00 AM. The wind will be from the WSW at 5.4 mph (8.6 kph).|
 The humidity will be 90%. The day will reach 73°F and the night will be 57°F, with 0% precipitation and a UV Index of 11.|


## Adding in memory

To give it memory we need to pass in a checkpointer. When passing in a checkpointer, we also have to pass in a thread_id when invoking the agent (so it knows which thread/conversation to resume from).

In [36]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [41]:
agent_executor = create_react_agent(model, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "new_abc123"}}

In [43]:
input_message = {"role": "user", "content": "Hi, I'm Bob!"}
print(f"config {config}")
for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

config {'configurable': {'thread_id': 'new_abc123'}}
================================ Human Message =================================

Hi, I'm Bob!
================================== Ai Message ==================================

Hello Bob! It's nice to meet you. How can I assist you today?


In [44]:
input_message = {"role": "user", "content": "What's my name?"}
for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's my name?
================================== Ai Message ==================================

Your name is Bob.


If you want to start a new conversation, all you have to do is change the thread_id used

In [45]:
config = {"configurable": {"thread_id": "xyz123"}}

input_message = {"role": "user", "content": "What's my name?"}
for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's my name?
================================== Ai Message ==================================

I do not know your name. I am a large language model, able to communicate in response to a wide range of prompts and questions, but I have no memory of past interactions and do not have access to any personal information about you.
